# Customer Churn – Data Cleaning

**Objective:** Clean the messy dataset by removing duplicates, handling missing values, and validating data types.

**Input:** `data/customer_churn_raw_v2.csv` (605 rows × 13 columns)  
**Output:** `data/customer_churn_clean.csv`

Steps: (1) inspect → (2) duplicates → (3) data types & placeholders → (4) outliers → (5) missing values → (6) standardize text → (7) validate → (8) export. Every decision is recorded in a **cleaning log** at the end.

In [1]:
import pandas as pd, numpy as np
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

RAW = 'data/customer_churn_raw_v2.csv'
CLEAN = 'data/customer_churn_clean.csv'
log = []   # (issue, rows affected, decision, reason)

df = pd.read_csv(RAW)
raw_rows = len(df)
print(df.shape)
df.head()

(605, 13)


,CustomerID,Age,Region,TenureMonths,SubscriptionType,MonthlyChargesINR,TotalChargesINR,ContractType,SupportTickets,LatePayments,PaymentMethod,AutoPay,Churn
0,CUST-100001,20,NaN,9,Pro,13097.76,unknown,Month-to-Month,1,2,Card,No,Yes
1,CUST-100002,55,Central,52,Pro,1101.18,57334.22,One Year,0,0,Card,Yes,No
2,CUST-100003,40,Central,67,Standard,692.66,44554.47,Month-to-Month,6,2,UPI,No,Yes
3,CUST-100004,53,South,48,Pro,1156.49,53956.12,One Year,2,2,Card,Yes,No
4,CUST-100005,21,West,24,Standard,754.16,18191.59,Month-to-Month,0,1,Card,Yes,No


## 1. Inspect the raw data

In [2]:
print(df.dtypes, '\n')
print('Missing values:\n', df.isna().sum()[lambda s: s > 0], '\n')
print('Duplicate rows:', df.duplicated().sum(), '| duplicate CustomerIDs:', df.CustomerID.duplicated().sum())
for col in ['Region','SubscriptionType','ContractType','PaymentMethod','AutoPay','Churn']:
    print(col, df[col].value_counts(dropna=False).to_dict())
df.describe().T

CustomerID               str
Age                    int64
Region                   str
TenureMonths           int64
SubscriptionType         str
MonthlyChargesINR    float64
TotalChargesINR          str
ContractType             str
SupportTickets         int64
LatePayments           int64
PaymentMethod            str
AutoPay                  str
Churn                    str
dtype: object 

Missing values:
 Region    13
dtype: int64 

Duplicate rows: 5 | duplicate CustomerIDs: 5
Region {'Central': 129, 'South': 117, 'West': 117, 'North': 117, 'East': 112, nan: 13}
SubscriptionType {'Basic': 212, 'Pro': 211, 'Standard': 182}
ContractType {'Month-to-Month': 318, 'One Year': 167, 'Two Year': 120}
PaymentMethod {'Wallet': 164, 'Card': 157, 'UPI': 143, 'Net Banking': 141}
AutoPay {'Yes': 353, 'No': 252}
Churn {'No': 355, 'Yes': 250}


,count,mean,std,min,25%,50%,75%,max
Age,605.0,42.578512,14.765469,18.00,30.00,42.00,55.00,68.00
TenureMonths,605.0,35.552066,20.948360,0.00,18.00,34.00,54.00,72.00
MonthlyChargesINR,605.0,868.016347,1104.807037,364.34,451.57,726.22,1103.47,13858.56
SupportTickets,605.0,1.757025,1.426617,0.00,1.00,2.00,3.00,6.00
LatePayments,605.0,1.277686,1.066768,0.00,0.00,1.00,2.00,4.00


**Findings**
- `Region` has 13 missing values.
- `TotalChargesINR` is stored as text because it contains the placeholder `'unknown'`.
- 5 exact duplicate records.
- `MonthlyChargesINR` has extreme values (up to ₹13,858) far above its plan's normal range.
- Tenure of 0 months exists (new customers).

## 2. Remove duplicates

In [3]:
n = df.duplicated().sum()
df = df.drop_duplicates(keep='first').reset_index(drop=True)
assert df.CustomerID.is_unique
log.append(('Duplicate records', n, 'Dropped (kept first occurrence)', 'Exact copies of the whole row; CustomerID must be unique'))
print(f'Dropped {n} duplicates -> {len(df)} rows')

Dropped 5 duplicates -> 600 rows


## 3. Fix data types & placeholder values

In [4]:
# 'unknown' is a placeholder for a missing value, not a real number
n_unknown = (df.TotalChargesINR == 'unknown').sum()
df['TotalChargesINR'] = pd.to_numeric(df.TotalChargesINR.replace('unknown', np.nan), errors='raise')
log.append(('TotalChargesINR text placeholder "unknown"', n_unknown, 'Converted to NaN, column cast to float', 'Needed to make the column numeric; imputed in step 5'))
df.dtypes

CustomerID               str
Age                    int64
Region                   str
TenureMonths           int64
SubscriptionType         str
MonthlyChargesINR    float64
TotalChargesINR      float64
ContractType             str
SupportTickets         int64
LatePayments           int64
PaymentMethod            str
AutoPay                  str
Churn                    str
dtype: object

## 4. Outliers in `MonthlyChargesINR`

Plan prices are tightly clustered (Basic ≈ ₹364–455, Standard ≈ ₹664–784, Pro ≈ ₹1,064–1,184), but 7 rows are ~10× higher. Let's check whether they're real or a systematic error.

In [5]:
cap = df.groupby('SubscriptionType').MonthlyChargesINR.transform(lambda s: s[s < 2000].max())
out = df[df.MonthlyChargesINR > cap * 1.5].copy()
out['monthly_div_12'] = (out.MonthlyChargesINR / 12).round(2)
out['total_div_tenure'] = (out.TotalChargesINR / out.TenureMonths).round(2)
out[['CustomerID','SubscriptionType','TenureMonths','MonthlyChargesINR','monthly_div_12','TotalChargesINR','total_div_tenure']]

,CustomerID,SubscriptionType,TenureMonths,MonthlyChargesINR,monthly_div_12,TotalChargesINR,total_div_tenure
0,CUST-100001,Pro,9,13097.76,1091.48,NaN,NaN
89,CUST-100090,Standard,42,9374.16,781.18,32698.72,778.54
178,CUST-100179,Pro,3,13239.48,1103.29,3189.06,1063.02
267,CUST-100268,Basic,65,5164.56,430.38,27928.48,429.67
356,CUST-100357,Standard,32,8863.80,738.65,22928.73,716.52
445,CUST-100446,Standard,57,8011.20,667.60,36568.18,641.55
534,CUST-100535,Pro,36,13858.56,1154.88,39825.11,1106.25


Dividing by 12 puts every outlier back inside its plan's normal range and matches `TotalChargesINR / TenureMonths`. So these are **annual amounts entered in the monthly column** (a unit error), not genuine high-value customers. **Decision: correct by ÷ 12** rather than drop (keeps the customer, and the Churn label is still valid).

In [6]:
mask = df.MonthlyChargesINR > cap * 1.5
df['MonthlyChargesFixed'] = mask            # flag kept for transparency
df.loc[mask, 'MonthlyChargesINR'] = (df.loc[mask, 'MonthlyChargesINR'] / 12).round(2)
log.append(('MonthlyChargesINR outliers (~12x plan price)', int(mask.sum()), 'Divided by 12 (annual -> monthly); flagged in monthly_charges_corrected', 'Value/12 falls inside plan range and agrees with TotalCharges/Tenure'))
print(mask.sum(), 'corrected'); print(df.groupby('SubscriptionType').MonthlyChargesINR.agg(['min','max']))

7 corrected
                      min      max
SubscriptionType                  
Basic              364.34   483.55
Pro               1064.16  1183.78
Standard           664.11   782.77


## 5. Handle missing values

**TotalChargesINR** – `TotalCharges ≈ MonthlyCharges × Tenure` for every complete row. Let me verify, then use that relationship to impute.

In [7]:
ok = df.TotalChargesINR.notna()
ratio = df.loc[ok & (df.TenureMonths > 0), 'TotalChargesINR'] / (df.loc[ok & (df.TenureMonths > 0), 'MonthlyChargesINR'] * df.loc[ok & (df.TenureMonths > 0), 'TenureMonths'])
print(ratio.describe().round(3))

count    586.000
mean       0.980
std        0.023
min        0.940
25%        0.961
50%        0.980
75%        0.998
max        1.020
dtype: float64


In [8]:
miss = df.TotalChargesINR.isna()
df.loc[miss, 'TotalChargesINR'] = (df.loc[miss, 'MonthlyChargesINR'] * df.loc[miss, 'TenureMonths']).round(2)
log.append(('TotalChargesINR missing', int(miss.sum()), 'Imputed = MonthlyCharges x Tenure (0 when Tenure = 0)', 'Ratio of actual/estimated is ~0.98 (range 0.9-1.02) so formula is reliable; mean/median would ignore tenure and be far worse'))
print(miss.sum(), 'imputed')

10 imputed


**Region** – it's categorical and nothing in the data reliably predicts it, so imputing the mode would invent information and skew regional churn analysis. **Decision: label as `Unknown`** (keeps all 13 rows).

In [9]:
n = df.Region.isna().sum()
df['Region'] = df.Region.fillna('Unknown')
log.append(('Region missing', int(n), 'Filled with "Unknown"', 'No reliable way to infer region; avoids biasing region-level churn and keeps rows'))
df.isna().sum().sum()

np.int64(0)

## 6. Standardize headers, categories & formats

In [10]:
# Headers -> snake_case
rename = {'CustomerID':'customer_id','Age':'age','Region':'region','TenureMonths':'tenure_months',
          'SubscriptionType':'subscription_type','MonthlyChargesINR':'monthly_charges_inr',
          'TotalChargesINR':'total_charges_inr','ContractType':'contract_type','SupportTickets':'support_tickets',
          'LatePayments':'late_payments','PaymentMethod':'payment_method','AutoPay':'auto_pay','Churn':'churn',
          'MonthlyChargesFixed':'monthly_charges_corrected'}
df = df.rename(columns=rename)

# Strip whitespace on text columns
txt = ['customer_id','region','subscription_type','contract_type','payment_method','auto_pay','churn']
for col in txt:
    df[col] = df[col].astype(str).str.strip()
df['customer_id'] = df.customer_id.str.upper()
# Category values were already consistent (verified in step 1), so no re-casing is needed

# Yes/No -> boolean-friendly 1/0 flags (keep readable originals too)
df['churn_flag'] = df.churn.map({'Yes':1,'No':0}).astype(int)
df['auto_pay'] = df.auto_pay.map({'Yes':True,'No':False})

# Dtypes
df = df.astype({'age':'int16','tenure_months':'int16','support_tickets':'int8','late_payments':'int8',
                'monthly_charges_inr':'float64','total_charges_inr':'float64'})
log.append(('Headers / text / types', len(df), 'snake_case headers, trimmed text, Yes/No -> bool/0-1, proper int/float dtypes', 'Consistency for analysis; added churn_flag (1 = churned) for modelling'))
print(df.contract_type.unique(), df.region.unique(), df.payment_method.unique())
df.dtypes

<StringArray>
['Month-to-Month', 'One Year', 'Two Year']
Length: 3, dtype: str <StringArray>
['Unknown', 'Central', 'South', 'West', 'North', 'East']
Length: 6, dtype: str <StringArray>
['Card', 'UPI', 'Wallet', 'Net Banking']
Length: 4, dtype: str


customer_id                      str
age                            int16
region                           str
tenure_months                  int16
subscription_type                str
monthly_charges_inr          float64
total_charges_inr            float64
contract_type                    str
support_tickets                 int8
late_payments                   int8
payment_method                   str
auto_pay                        bool
churn                            str
monthly_charges_corrected       bool
churn_flag                     int64
dtype: object

*(No date columns exist in this dataset, so there are no date formats to standardize.)*

## 7. Validate the clean data

In [11]:
assert df.customer_id.is_unique
assert df.isna().sum().sum() == 0
assert df.age.between(18, 100).all()
assert df.tenure_months.between(0, 120).all()
assert (df.monthly_charges_inr > 0).all() and (df.total_charges_inr >= 0).all()
assert df.churn.isin(['Yes','No']).all()
assert df.groupby('subscription_type').monthly_charges_inr.max().max() < 1300
chk = df[df.tenure_months > 0]
assert (chk.total_charges_inr / (chk.monthly_charges_inr * chk.tenure_months)).between(0.85, 1.1).all()
print('All validation checks passed')
print(f'Rows: {raw_rows} raw -> {len(df)} clean')

All validation checks passed
Rows: 605 raw -> 600 clean


## 8. Export

In [12]:
df.to_csv(CLEAN, index=False)
pd.DataFrame(log, columns=['Issue','Rows affected','Decision','Reason']).to_csv('data/cleaning_log.csv', index=False)
print('Saved', CLEAN)
df.head()

Saved

 data/customer_churn_clean.csv


,customer_id,age,region,tenure_months,subscription_type,monthly_charges_inr,total_charges_inr,contract_type,support_tickets,late_payments,payment_method,auto_pay,churn,monthly_charges_corrected,churn_flag
0,CUST-100001,20,Unknown,9,Pro,1091.48,9823.32,Month-to-Month,1,2,Card,False,Yes,True,1
1,CUST-100002,55,Central,52,Pro,1101.18,57334.22,One Year,0,0,Card,True,No,False,0
2,CUST-100003,40,Central,67,Standard,692.66,44554.47,Month-to-Month,6,2,UPI,False,Yes,False,1
3,CUST-100004,53,South,48,Pro,1156.49,53956.12,One Year,2,2,Card,True,No,False,0
4,CUST-100005,21,West,24,Standard,754.16,18191.59,Month-to-Month,0,1,Card,True,No,False,0


## Cleaning log (every decision)

In [13]:
pd.set_option('display.max_colwidth', 120)
pd.DataFrame(log, columns=['Issue','Rows affected','Decision','Reason'])

,Issue,Rows affected,Decision,Reason
0,Duplicate records,5,Dropped (kept first occurrence),Exact copies of the whole row; CustomerID must be unique
1,"TotalChargesINR text placeholder ""unknown""",10,"Converted to NaN, column cast to float",Needed to make the column numeric; imputed in step 5
2,MonthlyChargesINR outliers (~12x plan price),7,Divided by 12 (annual -> monthly); flagged in monthly_charges_corrected,Value/12 falls inside plan range and agrees with TotalCharges/Tenure
3,TotalChargesINR missing,10,Imputed = MonthlyCharges x Tenure (0 when Tenure = 0),Ratio of actual/estimated is ~0.98 (range 0.9-1.02) so formula is reliable; mean/median would ignore tenure and be f...
4,Region missing,13,"Filled with ""Unknown""",No reliable way to infer region; avoids biasing region-level churn and keeps rows
5,Headers / text / types,600,"snake_case headers, trimmed text, Yes/No -> bool/0-1, proper int/float dtypes",Consistency for analysis; added churn_flag (1 = churned) for modelling


## Summary

| | Before | After |
|---|---|---|
| Rows | 605 | 600 |
| Duplicates | 5 | 0 |
| Missing / placeholder values | 13 Region + 10 `unknown` TotalCharges | 0 |
| Monthly-charge outliers | 7 (≈12× too high) | 0 (corrected) |
| `TotalChargesINR` dtype | text | float |

The cleaned file is ready for the next task (churn rate, CLV, behavioural drivers).